# Process to be used by frontend code

Initial iteration to plot clusters on UI from Google Colab outputs

In [41]:
import ast
import json
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

In [51]:
def process_cluster_file(filepath):
    with open(filepath) as f:
        data = json.load(f)

    all_artist_rows = []
    all_cluster_positions = []

    for version in data:
        n = version["n"]
        n_clusters = len(version["clusters"])

        rows = []
        for cluster in version["clusters"]:
            for point in cluster["data"]:
                row = {
                    "n": n,
                    "cluster": int(cluster["key"]),
                    "keywords": cluster["keywords"],
                }
                row.update(point)
                rows.append(row)

        df = pd.DataFrame(rows)

        keyword_map = {int(c["key"]): c["keywords"] for c in version["clusters"]}

        center_vectors = np.array(
            [
                [
                    df[df["cluster"] == c][f"dist_to_cluster_{cc}"].mean()
                    for cc in range(n_clusters)
                ]
                for c in range(n_clusters)
            ]
        )
        centers_2d = PCA(n_components=2).fit_transform(center_vectors)
        centers_2d -= centers_2d.min(axis=0)
        denom = centers_2d.max(axis=0)
        denom[denom == 0] = 1
        centers_2d = centers_2d / denom

        for artist_id, group in df.groupby("artist_id"):
            total_points = len(group)
            cluster_counts = group["cluster"].value_counts()
            primary_cluster = int(cluster_counts.idxmax())

            cluster_distribution = {
                str(c): round(cluster_counts.get(c, 0) / total_points, 4)
                for c in range(n_clusters)
            }

            avg_dists = {}
            for c in range(n_clusters):
                key = f"avg_dist_to_cluster_{c}"
                mean_val = group[f"dist_to_cluster_{c}"].mean()
                avg_dists[key] = None if pd.isna(mean_val) else round(float(mean_val), 4)

            closest_by_dist = int(
                np.argmin(
                    [
                        avg_dists[f"avg_dist_to_cluster_{c}"]
                        if avg_dists[f"avg_dist_to_cluster_{c}"] is not None
                        else np.inf
                        for c in range(n_clusters)
                    ]
                )
            )

            all_artist_rows.append(
                {
                    "n": n,
                    "artist": artist_id,
                    "total_points": total_points,
                    "primary_cluster": primary_cluster,
                    "closest_cluster_by_dist": closest_by_dist,
                    "cluster_distribution": cluster_distribution,
                    **avg_dists,
                }
            )

        all_cluster_positions.append(
            {
                "n": n,
                "cluster_positions": {
                    str(c): {
                        "x": round(float(centers_2d[c][0]), 4),
                        "y": round(float(centers_2d[c][1]), 4),
                        "keywords": keyword_map.get(c, []),
                    }
                    for c in range(n_clusters)
                },
            }
        )

    return all_artist_rows, all_cluster_positions

In [43]:
def _assign_ring_positions(layer_items, layer_radius):
    for layer_index, items in layer_items.items():
        if not items:
            continue

        m = len(items)
        phase = layer_index * (np.pi / max(m, 6))
        for j, row in enumerate(items):
            row["_angle"] = round(float((j / max(m, 1)) * 2 * np.pi + phase), 4)
            row["_radius_frac"] = round(float(layer_radius(layer_index)), 4)


def add_ring_layout(rows):
    by_n = defaultdict(list)
    for row in rows:
        by_n[row["n"]].append(row)

    for _, grouped_rows in by_n.items():
        by_cluster = defaultdict(list)
        for row in grouped_rows:
            by_cluster[row["primary_cluster"]].append(row)

        for c, cluster_rows in by_cluster.items():
            dist_key = f"avg_dist_to_cluster_{c}"
            sorted_rows = sorted(cluster_rows, key=lambda r: r.get(dist_key, 0.5))
            n_items = len(sorted_rows)

            layer_count = min(5, max(2, int(np.ceil(np.sqrt(n_items)))))
            per_layer = int(np.ceil(n_items / layer_count))
            r_min_frac, r_max_frac = 0.16, 1.0

            layer_items = defaultdict(list)
            for layer in range(layer_count):
                layer_slice = sorted_rows[layer * per_layer : (layer + 1) * per_layer]
                for row in layer_slice:
                    layer_items[layer].append(row)

            def _artist_radius(layer_index):
                frac = layer_index / max(layer_count - 1, 1)
                return r_min_frac + frac * (r_max_frac - r_min_frac)

            _assign_ring_positions(layer_items, _artist_radius)

    return rows


def add_exhibition_ring_layout(items, r_inner=0.35, r_outer=0.85):
    sorted_items = sorted(items, key=lambda r: (r["isShared"], r["artistId"]))

    layer_items = defaultdict(list)
    for row in sorted_items:
        layer = 1 if row["isShared"] else 0
        layer_items[layer].append(row)

    def _exhibit_radius(layer_index):
        return r_outer if layer_index == 1 else r_inner

    _assign_ring_positions(layer_items, _exhibit_radius)
    return sorted_items

In [52]:
REFERENCE_W = 1200
REFERENCE_H = 800
CANVAS_PADDING = 220
CLUSTER_RADIUS_BASE = 210
COLLISION_GAP = 40


def resolve_cluster_positions(positions_data, summary_data):
    counts = defaultdict(lambda: defaultdict(int))
    for row in summary_data:
        counts[row["n"]][row["primary_cluster"]] += 1

    resolved = []

    for entry in positions_data:
        n = entry["n"]
        cluster_positions = entry["cluster_positions"]
        n_clusters = len(cluster_positions)

        scale = max(1.0, 1 + (n - 6) * 0.18)
        ref_w = int(REFERENCE_W * scale)
        ref_h = int(REFERENCE_H * scale)
        gap = COLLISION_GAP + max(0, (n - 6) * 12)

        inner_w = ref_w - CANVAS_PADDING * 2
        inner_h = ref_h - CANVAS_PADDING * 2

        pos = []
        for c in range(n_clusters):
            p = cluster_positions.get(str(c), {"x": 0.5, "y": 0.5, "keywords": []})
            artist_count = counts[n].get(c, 1)
            footprint = min(CLUSTER_RADIUS_BASE, 160 + min(52, artist_count * 5))
            pos.append(
                {
                    "c": c,
                    "x": CANVAS_PADDING + float(p["x"]) * inner_w,
                    "y": CANVAS_PADDING + float(p["y"]) * inner_h,
                    "r": footprint,
                    "keywords": p.get("keywords", []),
                }
            )

        iterations = 300 + max(0, (n - 6) * 80)
        for _ in range(iterations):
            for i in range(n_clusters):
                for j in range(i + 1, n_clusters):
                    dx = pos[j]["x"] - pos[i]["x"]
                    dy = pos[j]["y"] - pos[i]["y"]
                    d = max(np.hypot(dx, dy), 0.001)
                    need = pos[i]["r"] + pos[j]["r"] + gap
                    if d < need:
                        push = (need - d) / 2
                        pos[i]["x"] -= (dx / d) * push
                        pos[i]["y"] -= (dy / d) * push
                        pos[j]["x"] += (dx / d) * push
                        pos[j]["y"] += (dy / d) * push

        all_x = [p["x"] for p in pos]
        all_y = [p["y"] for p in pos]
        margin = 80
        canvas_min_w = int(max(ref_w, max(all_x) + CANVAS_PADDING + margin))
        canvas_min_h = int(max(ref_h, max(all_y) + CANVAS_PADDING + margin))

        resolved_positions = {}
        for p in pos:
            nx = np.clip((p["x"] - CANVAS_PADDING) / inner_w, 0.02, 0.98)
            ny = np.clip((p["y"] - CANVAS_PADDING) / inner_h, 0.02, 0.98)
            resolved_positions[str(p["c"])] = {
                "x": round(float(nx), 4),
                "y": round(float(ny), 4),
                "keywords": p["keywords"],
            }

        resolved.append(
            {
                "n": n,
                "cluster_positions": resolved_positions,
                "canvas_min_w": canvas_min_w,
                "canvas_min_h": canvas_min_h,
                "aspect_ratio": round(canvas_min_w / canvas_min_h, 4),
            }
        )

    return resolved

In [45]:
def write_json(path, data):
    with open(path, "w") as f:
        json.dump(data, f, indent=2, allow_nan=False)

## Artist/institution cluster outputs

In [53]:
institution_artist_rows, institution_positions = process_cluster_file(
    "../../data/colab/inst_clusters.json"
)
artist_artist_rows, artist_positions = process_cluster_file(
    "../../data/colab/ar_clusters.json"
)

institution_artist_rows = add_ring_layout(institution_artist_rows)
artist_artist_rows = add_ring_layout(artist_artist_rows)

institution_positions = resolve_cluster_positions(
    institution_positions,
    institution_artist_rows,
)
artist_positions = resolve_cluster_positions(
    artist_positions,
    artist_artist_rows,
)

In [54]:
write_json("../../data/ui/institution_clusters.json", institution_artist_rows)
write_json("../../data/ui/artist_clusters.json", artist_artist_rows)
write_json("../../data/ui/institution_cluster_positions.json", institution_positions)
write_json("../../data/ui/artist_cluster_positions.json", artist_positions)

## Exhibition layout output

In [48]:
df = pd.read_csv("../../data/colab/exhibitions_with_labels.csv")
df["artist_ids"] = df["artists"].apply(ast.literal_eval)

all_artist_occurrences = Counter()
for ids in df["artist_ids"]:
    for aid in ids:
        all_artist_occurrences[aid] += 1

R_INNER = 0.35
R_OUTER = 0.85

exhibitions_out = []

for _, row in df.iterrows():
    artist_ids = row["artist_ids"]
    labels = ast.literal_eval(row["labels"]) if isinstance(row["labels"], str) else row["labels"]

    items = [
        {
            "artistId": aid,
            "isShared": all_artist_occurrences[aid] > 1,
        }
        for aid in artist_ids
    ]

    items = add_exhibition_ring_layout(items, r_inner=R_INNER, r_outer=R_OUTER)

    exhibitions_out.append(
        {
            "id": int(row["id"]),
            "name": row["name"],
            "labels": labels[:3],
            "items": items,
        }
    )

write_json("../../data/ui/exhibition_clusters.json", exhibitions_out)

In [50]:
# new exhibitions format dedupe artist points
df = pd.read_csv("../../data/colab/exhibitions_with_labels.csv")
df["artist_ids"] = df["artists"].apply(ast.literal_eval)

def parse_labels(x):
    if isinstance(x, str):
        try: return ast.literal_eval(x)
        except: return []
    return x or []

df["labels_parsed"] = df["labels"].apply(parse_labels)

# Artist → which exhibitions they appear in
artist_exhibitions = {}
for _, row in df.iterrows():
    for aid in row["artist_ids"]:
        artist_exhibitions.setdefault(aid, [])
        artist_exhibitions[aid].append(int(row["id"]))

R_INNER = 0.35
R_OUTER = 0.85

# Assign angles to exclusive artists per exhibition
exclusive_by_exhibition = {}
for aid, eids in artist_exhibitions.items():
    if len(eids) == 1:
        exclusive_by_exhibition.setdefault(eids[0], []).append(aid)

exclusive_angle_map = {}
for eid, aids in exclusive_by_exhibition.items():
    for j, aid in enumerate(sorted(aids)):
        exclusive_angle_map[aid] = round((j / max(len(aids), 1)) * 2 * np.pi, 4)

# Assign jitter angles to shared artists so they don't pile up at the same midpoint
shared_artists = [aid for aid, eids in artist_exhibitions.items() if len(eids) > 1]
for j, aid in enumerate(shared_artists):
    # Space them evenly so multiple shared artists don't overlap
    pass  # angle assigned below per-artist

# Build output
exhibitions_out = []
for _, row in df.iterrows():
    exhibitions_out.append({
        "id": int(row["id"]),
        "name": row["name"],
        "labels": row["labels_parsed"][:3],
    })

artists_out = []
for j, (aid, eids) in enumerate(artist_exhibitions.items()):
    is_shared = len(eids) > 1
    entry = {
        "artistId": aid,
        "exhibitionIds": eids,
        "isShared": is_shared,
        "_radius_frac": R_OUTER if is_shared else R_INNER,
    }
    if not is_shared:
        entry["_angle"] = exclusive_angle_map.get(aid, 0.0)
    else:
        # Jitter angle spreads shared artists around their midpoint
        entry["_jitter_angle"] = round((j / max(len(shared_artists), 1)) * 2 * np.pi, 4)
    artists_out.append(entry)

output_object = {"exhibitions": exhibitions_out, "artists": artists_out}

write_json("../../data/ui/exhibition_clusters.json", output_object)